In [11]:
import os

In [62]:
from typing import TypedDict, Annotated, Sequence, Optional
from dotenv import load_dotenv

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from langchain_core.messages import (
    BaseMessage,
    HumanMessage,
    SystemMessage,
    AIMessage,
    ToolMessage
)
from langchain_core.tools import tool

from utils.clients.hcp_clients import get_openai_chat_client as chat_client
#from IPython.display import Image, display
import logging
import json

load_dotenv()
logging.basicConfig(level=logging.INFO)

# Define structure of the state
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]


class EnricherAgent(StateGraph[AgentState]):
    def __init__(self):

        # initialize the LLM
        self.__llm = chat_client(
            model="gpt-4o",
            temperature=0.7,
            max_retries=5
        )

    
    def user_msg_enricher(self, state: AgentState) -> AgentState:
        """
        Enriches the user message with additional context.
        """

        llm_enricher = self.__llm

        systemPrompt = """
        # Care Staff Query Enricher Agent

        ## Role  
        You are an AI assistant that helps healthcare professionals by enriching their queries with relevant context.  
        You are a **Query Enricher Agent** supporting care staff in a healthcare and social care environment.  

        Staff may enter shorthand, fragmented, or vague queries when searching care records, incident logs, or compliance documents.  
        Your job is to **rewrite these into clear, professional, context-rich queries phrased as questions** that can be understood by a search or retrieval system.  

        ---

        ## Enrichment Rules  
        - Expand shorthand into full, professional wording.  
        - Clarify vague or incomplete queries into **full questions** while keeping intent intact.  
        - Assume the query relates to **resident care, staff activity, or compliance records** unless specified otherwise.  
        - Maintain neutrality, professionalism, and respectful language.  
        - If a query is ambiguous, provide multiple possible enriched questions or clearly note the ambiguity.  
        - Do not add details that were not implied.  
        - Always phrase the final enriched query as a **question**. 

        Examples:
        User Query: "med admin 2pm"
        Enriched Query: "What are the details of medication administration at 2 PM for all residents?"

        User Query: "fall risk for AS"
        Enriched Query: "What are the assessments and management strategies for fall risk for resident AS?"

        User Query: "incident log"
        Enriched Query: "What are the records of all incident logs involving residents or staff within the care facility?"

        User Query: "meds procedure for EB"
        Enriched Query: "What are the details of the medication administration procedure for resident EB?"

        User Query: "shift handover"
        Enriched Query: "What are the procedures and notes related to shift handovers among care staff?"

        User Query: "Known allergies for John"
        Enriched Query: "What are the known allergies for resident John?"

        User Query: "pressure sore treatment for BG"
        Enriched Query: "What are the details of pressure sore treatment for resident BG?"

        User Query: "How do I administer Cosmcol for ER?"
        Enriched Query: "What are the instructions for administering Cosmcol for resident ER?"

        Finally, respond ONLY with a valid JSON object in this format. Do NOT use triple backticks, do NOT use the word 'json', and do NOT add any extra text or formatting. Your reply should look exactly like this:
        {{
            "enriched_query": "<Your enriched query here>"
        }}

        Now, enrich the following user query:
        {{user_query}}
        """

        # Inject the latest user query into the system prompt
        user_query = state['messages'][-1].content if state['messages'] else ""
        systemPrompt = systemPrompt.replace("{user_query}", user_query)

        try:
            print("=============State Before Enrichment================")
            print(state['messages'][-1])
            response = llm_enricher.invoke([systemPrompt] + state['messages'])
            print(response.content)
            response = AIMessage(content=response.content)
            logging.info("Enriched query: %s", response.content)
            state['messages'] = state['messages'] + [response]
            print("=============Last Message in the State================")
            print(state['messages'][-1])
        except Exception as e:
            logging.error("Error during message enrichment: %s", str(e))

            state['messages'] = state['messages'] + [AIMessage(content=f"Error: {str(e)}")]
        
        return state
        #return {"messages": state['messages']}
    

    # Define the graph structure
    def create_enricher_agent(self) -> StateGraph:
        graph = StateGraph(AgentState)
        
        # Define nodes
        graph.add_node("enrich_user_message_agent", self.user_msg_enricher)

        # Construct Flow
        graph.add_edge(START, "enrich_user_message_agent")
        graph.add_edge("enrich_user_message_agent", END)

        # Compile the graph
        agent_graph = graph.compile()
        return agent_graph
    
    def visualize_graph(self, xray: bool = True) -> Optional[str]:
        """
        Returns the Mermaid diagram string for the agent graph.
        """
        graph = self.create_enricher_agent()
        try:
            return graph.get_graph(xray=xray)
        except Exception as e:
            logging.error("Error generating graph visualization: %s", str(e))
            return None

In [63]:
from IPython.display import display, Markdown, Image

agent = EnricherAgent()

# Generate LangGraph visualization
graphdiagram = agent.visualize_graph()

# Export Mermaid code
mermaid_diagram = graphdiagram.draw_mermaid()
if mermaid_diagram:
    with open("graph_enricher_agent.md", "w") as f:
        f.write(f"```mermaid\n{mermaid_diagram}\n```")
    display(Markdown("Mermaid code saved to **graph.md** (open with Markdown preview for rendering)."))
    display(Markdown(f"```mermaid\n{mermaid_diagram}\n```"))
else:
    print("Could not generate Mermaid diagram.")

# Export PNG
# graphdiagram.draw_png("graph.png")

# Display PNG inline in notebook
# display(Image("graph.png"))

Mermaid code saved to **graph.md** (open with Markdown preview for rendering).

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	enrich_user_message_agent(enrich_user_message_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> enrich_user_message_agent;
	enrich_user_message_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [9]:
from IPython.display import display, Markdown

agent = EnricherAgent()
mermaid_diagram = agent.visualize_graph().draw_mermaid()

#LangGraph direct export
# graphdiagram = agent.visualize_graph()
# graphdiagram.draw_png("graph.png")

if mermaid_diagram:
    with open("graph.md", "w") as f:
        f.write(f"```mermaid\n{mermaid_diagram}\n```")
    display(Markdown(f"```mermaid\n{mermaid_diagram}\n```"))
else:
    print("Could not generate Mermaid diagram.")

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	enrich_user_message_agent(enrich_user_message_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> enrich_user_message_agent;
	enrich_user_message_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [ ]:
if __name__ == "__main__":
    agent = EnricherAgent()
    enricher_agent = agent.create_enricher_agent()
    while True:
        user_input = input("Enter your query (or 'exit' to quit): ")
        if user_input.lower() == 'exit':
            break
        else:
            result = enricher_agent.invoke({"messages": [HumanMessage(content=user_input)]})
            print(result)
            enriched_message = result['messages'][-1]
            enriched = json.loads(enriched_message.content)
            query_for_next_llm = enriched["enriched_query"]

            print("=============Enriched Query================")
            print(f"Enriched Query: {query_for_next_llm}\n")


=============State Before Enrichment================
content="What is BG's medications" additional_kwargs={} response_metadata={} id='77e16abb-32b6-4354-8089-3ee9a36347ce'


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Enriched query: {
    "enriched_query": "What are the current medications prescribed for resident BG?"
}


{
    "enriched_query": "What are the current medications prescribed for resident BG?"
}
=============Last Message in the State================
content='{\n    "enriched_query": "What are the current medications prescribed for resident BG?"\n}' additional_kwargs={} response_metadata={}
{'messages': [HumanMessage(content="What is BG's medications", additional_kwargs={}, response_metadata={}, id='77e16abb-32b6-4354-8089-3ee9a36347ce'), AIMessage(content='{\n    "enriched_query": "What are the current medications prescribed for resident BG?"\n}', additional_kwargs={}, response_metadata={}, id='e4f044c6-6603-4ee4-b3a0-d345df27bbc3')]}
=============Enriched Query================
Enriched Query: What are the current medications prescribed for resident BG?

=============State Before Enrichment================
content='care plan fpr pmc' additional_kwargs={} response_metadata={} id='8a8b1bef-2f71-4fc7-81e7-af46ca59bfd8'


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Enriched query: {
    "enriched_query": "What are the details of the care plan for resident PMC?"
}


{
    "enriched_query": "What are the details of the care plan for resident PMC?"
}
=============Last Message in the State================
content='{\n    "enriched_query": "What are the details of the care plan for resident PMC?"\n}' additional_kwargs={} response_metadata={}
{'messages': [HumanMessage(content='care plan fpr pmc', additional_kwargs={}, response_metadata={}, id='8a8b1bef-2f71-4fc7-81e7-af46ca59bfd8'), AIMessage(content='{\n    "enriched_query": "What are the details of the care plan for resident PMC?"\n}', additional_kwargs={}, response_metadata={}, id='58224a28-f67f-4c58-85bd-3ef796287227')]}
=============Enriched Query================
Enriched Query: What are the details of the care plan for resident PMC?

=============State Before Enrichment================
content='What was my first question' additional_kwargs={} response_metadata={} id='1e66fcf6-0548-42fb-8e90-6c2f158bce03'


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Enriched query: {
    "enriched_query": "What is the content of the first query I submitted?"
}


{
    "enriched_query": "What is the content of the first query I submitted?"
}
=============Last Message in the State================
content='{\n    "enriched_query": "What is the content of the first query I submitted?"\n}' additional_kwargs={} response_metadata={}
{'messages': [HumanMessage(content='What was my first question', additional_kwargs={}, response_metadata={}, id='1e66fcf6-0548-42fb-8e90-6c2f158bce03'), AIMessage(content='{\n    "enriched_query": "What is the content of the first query I submitted?"\n}', additional_kwargs={}, response_metadata={}, id='beb2d4aa-7b6d-452b-8f63-cf1648e3f877')]}
=============Enriched Query================
Enriched Query: What is the content of the first query I submitted?

=============State Before Enrichment================
content='quit' additional_kwargs={} response_metadata={} id='3121d7cb-ba34-4d2d-ab59-dc8c1bcb2dda'


INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Enriched query: {
    "enriched_query": "What are the protocols for staff when a resident decides to quit a treatment or program?"
}


{
    "enriched_query": "What are the protocols for staff when a resident decides to quit a treatment or program?"
}
=============Last Message in the State================
content='{\n    "enriched_query": "What are the protocols for staff when a resident decides to quit a treatment or program?"\n}' additional_kwargs={} response_metadata={}
{'messages': [HumanMessage(content='quit', additional_kwargs={}, response_metadata={}, id='3121d7cb-ba34-4d2d-ab59-dc8c1bcb2dda'), AIMessage(content='{\n    "enriched_query": "What are the protocols for staff when a resident decides to quit a treatment or program?"\n}', additional_kwargs={}, response_metadata={}, id='d0d457c6-9fbf-48d0-9cbd-71e0a021b095')]}
=============Enriched Query================
Enriched Query: What are the protocols for staff when a resident decides to quit a treatment or program?



In [82]:
from typing import TypedDict, List, Optional, Annotated, Sequence
from dotenv import load_dotenv
import logging

from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage

from utils.clients.hcp_clients import get_openai_chat_client as chat_client

from IPython.display import display, Markdown

# Setup
logging.basicConfig(level=logging.INFO)
load_dotenv()

# Define structure of the agent state
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]
    selected_agent: Optional[str]

# RoutingAgent definition
class RoutingAgent(StateGraph[AgentState]):

    def __init__(self, agent_details: List[dict]):

        self.agent_details = agent_details

        # Initialize LLM
        self.__llm = chat_client(
            model="gpt-4o",
            temperature=0.0,
            max_retries=5
        )

    def route_query(self, state: AgentState) -> AgentState:
        """
        Routes the enriched query to the appropriate specialist agent.
        """
        system_prompt = f"""
        You are a healthcare query routing agent.

        Your role is to analyze the user’s query and determine which specialist agent(s) should handle it. 
        You must decide based on the available agents’ expertise. 

        Rules:
        1. If the query clearly belongs to a single specialist agent’s domain, return that agent’s name.
        2. If the query requires collaboration between multiple specialist agents, return ALL relevant agent names as a comma-separated list in the correct execution order.
        - For example: if a query first requires retrieving data (Incident Log Agent) and then checking rules (Compliance Document Agent), return: "Incident Log Agent, Compliance Document Agent".
        3. If the query is not healthcare-related or does not match any agent, return "None".
        4. Do not include any explanation — return only the agent name(s).

        The available specialist agents are:
        {self.agent_details}

        Example:
        User Query: "Show me Mrs. Smith’s last incident report and check if it matches compliance rules."
        Expected Output: "Incident Log Agent, Compliance Document Agent"
        User Query: "What are the known allergies for resident John?"
        Expected Output: "Care Record Agent"
        """

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(
                content=f"Enriched Query: {state['messages'][-1].content}\n\n"
                        "Which agent is best suited to handle this query? Respond with the agent name only, or 'None'."
            )
        ]

        try:
            response = self.__llm.invoke(messages)
            selected_agent = response.content.strip()
            logging.info(f"Routing Agent selected: {selected_agent}")
            state['selected_agent'] = selected_agent
        except Exception as e:
            logging.error(f"Error during routing: {e}")
            state['selected_agent'] = "None"

        return state

    def create_routing_graph(self, agent_details: List[dict]) -> StateGraph:
        """
        Creates the LangGraph nodes and edges for routing.
        """
        graph = StateGraph(AgentState)
        graph.add_node("route_query", self.route_query)
        graph.set_entry_point("route_query")
        graph.add_edge("route_query", END)

        agent_graph = graph.compile()

        
        logging.info("Routing graph created and compiled.")
        return agent_graph

    def visualize_graph(self, agent_details: List[dict], xray: bool = True) -> Optional[str]:
        """
        Generates a Mermaid diagram string for the routing agent graph.
        """
        try:
            # Create and compile the graph
            compiled_graph = self.create_routing_graph(agent_details)
            # Use compiled graph to generate Mermaid code
            return compiled_graph.get_graph(xray=xray)
        except Exception as e:
            logging.error(f"Error generating graph visualization: {e}")
            return None



In [83]:
# Define agent details
agent_details = [
    {"name": "Care Record Agent", "expertise": "Handling queries related to resident care records."},
    {"name": "Incident Log Agent", "expertise": "Managing queries about incident logs and reports."},
    {"name": "Compliance Document Agent", "expertise": "Dealing with queries regarding compliance documents and regulations."}
]

# Initialize agent
agent = RoutingAgent(agent_details)

# Generate Mermaid diagram
mermaid_diagram = agent.visualize_graph(agent_details).draw_mermaid()

if mermaid_diagram:
    with open("graph_route_agent.md", "w") as f:
        f.write(f"```mermaid\n{mermaid_diagram}\n```")
    display(Markdown("Mermaid code saved to **graph_route_agent.md**"))
    display(Markdown(f"```mermaid\n{mermaid_diagram}\n```"))
else:
    print("Could not generate Mermaid diagram.")


INFO:root:Routing graph created and compiled.


Mermaid code saved to **graph_route_agent.md**

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	route_query(route_query)
	__end__([<p>__end__</p>]):::last
	__start__ --> route_query;
	route_query --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

In [84]:
if __name__ == "__main__":

    agent_details = [
        {"name": "Care Record Agent", "expertise": "Handling queries related to resident care records. Such as medication administration, care plans, allergies, and health conditions."},
        {"name": "Incident Log Agent", "expertise": "Managing queries about incident logs and reports. Such as falls, injuries, behavioral incidents, and safety concerns."},
        {"name": "Compliance Document Agent", "expertise": "Dealing with queries regarding compliance documents and regulations. Such as health and safety protocols, data protection policies, and care standards."}
    ]

    # Initialize agent
    agent = RoutingAgent(agent_details)

    routing_agent = agent.create_routing_graph(agent_details)
    while True:
        user_input = input("Enter Query: ")
        if user_input.lower() == 'exit':
            break
        else:
            result = routing_agent.invoke({"messages": [HumanMessage(content=user_input)]})
            print(result)
            print("=============Selected Agent================")
            print(f"Selected Agent: {result['selected_agent']}\n")


INFO:root:Routing graph created and compiled.
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Routing Agent selected: Care Record Agent


{'messages': [HumanMessage(content='What medications is Mrs. Smith currently prescribed?', additional_kwargs={}, response_metadata={}, id='c00eec99-b1a9-4da0-915b-2bf5e6bc2961')], 'selected_agent': 'Care Record Agent'}
=============Selected Agent================
Selected Agent: Care Record Agent



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Routing Agent selected: Care Record Agent


{'messages': [HumanMessage(content='Show me John Doe’s last care plan update.', additional_kwargs={}, response_metadata={}, id='bb087281-4a6f-4728-82f3-7e100af62dc2')], 'selected_agent': 'Care Record Agent'}
=============Selected Agent================
Selected Agent: Care Record Agent



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Routing Agent selected: Care Record Agent


...

{'messages': [HumanMessage(content='List all incidents involving medication errors in the past month', additional_kwargs={}, response_metadata={}, id='1c8f1d37-ee03-418e-b1aa-fd47ae2b16a0')], 'selected_agent': 'Incident Log Agent'}
=============Selected Agent================
Selected Agent: Incident Log Agent



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Routing Agent selected: Incident Log Agent, Compliance Document Agent


{'messages': [HumanMessage(content='Compare the incident report for Mr. Green with compliance guidelines for escalation.', additional_kwargs={}, response_metadata={}, id='147712a1-1460-4756-ac32-7bb68f90a596')], 'selected_agent': 'Incident Log Agent, Compliance Document Agent'}
=============Selected Agent================
Selected Agent: Incident Log Agent, Compliance Document Agent



INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:Routing Agent selected: Care Record Agent, Compliance Document Agent


{'messages': [HumanMessage(content='Show Mrs. Taylor’s dietary and verify that it aligns with nutritional compliance regulations.', additional_kwargs={}, response_metadata={}, id='096c8596-0a42-4775-9acd-b039af5cca99')], 'selected_agent': 'Care Record Agent, Compliance Document Agent'}
=============Selected Agent================
Selected Agent: Care Record Agent, Compliance Document Agent

